# Breast Cancer Survival Analysis
## Notebook 0 — Data Cleaning

This notebook takes the two raw datasets and produces `clean_data_breast_cancer.xlsx` — the single file used by all subsequent notebooks. Every cleaning decision is documented here so the pipeline is fully reproducible from raw data.

**Input files (place in `data/`):**
- `SEER_Breast_Cancer_Dataset_.csv` — semicolon-separated, 4024 rows, 15 columns
- `METABRIC_raw.csv` — tab/comma-separated, downloaded from cBioPortal

**Output:** `data/clean_data_breast_cancer.xlsx` with two sheets: `SEER` and `METABRIC`

---

### Contents
1. [Setup](#1-setup)
2. [SEER Cleaning](#2-seer-cleaning)
3. [METABRIC Cleaning](#3-metabric-cleaning)
4. [Validation](#4-validation)
5. [Export](#5-export)

---
## 1. Setup

In [14]:
import os
import pandas as pd
import numpy as np

# ── Working directory ──────────────────────────────────
try:
    notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    notebook_dir = os.path.abspath('')
os.chdir(notebook_dir)
#print('Working directory:', os.getcwd())

# ── Paths ────────────────────────────────────────
SEER_PATH             = 'data/SEER Breast Cancer Dataset .csv'
METABRIC_PATIENT_PATH = 'data/data_clinical_patient.xlsx'
METABRIC_SAMPLE_PATH  = 'data/data_clinical_sample.txt'
OUTPUT_PATH           = 'data/clean_data_breast_cancer.xlsx'

# ── Target schema ──────────────────────────────────────────────────────
FINAL_COLUMNS = [
    'Age',                # continuous — age at diagnosis
    'T Stage',            # ordinal 1-4 — tumour spread
    'Grade',              # ordinal 1-3 — cell abnormality
    'Tumor Size',         # continuous — mm
    'Estrogen Status',    # binary 0/1 — receptor positive
    'Progesterone Status',# binary 0/1 — receptor positive
    'Regional Node 1',    # discrete — number of positive nodes
    'Survival Months',    # continuous — months after diagnosis
    'Status',             # binary 0=Dead / 1=Alive
]

print('Target schema defined:', FINAL_COLUMNS)

Target schema defined: ['Age', 'T Stage', 'Grade', 'Tumor Size', 'Estrogen Status', 'Progesterone Status', 'Regional Node 1', 'Survival Months', 'Status']


---
## 2. SEER Cleaning

### 2.1 Load raw data

In [15]:
# SEER is semicolon-separated — standard CSV read would produce a single column
seer_raw = pd.read_csv(SEER_PATH, sep=';')
print(f'Raw SEER: {seer_raw.shape[0]:,} rows x {seer_raw.shape[1]} columns')
print(f'Columns: {list(seer_raw.columns)}')
seer_raw.head(3)

Raw SEER: 4,024 rows x 15 columns
Columns: ['Age', 'Race', 'Marital Status', 'T Stage', 'N Stage', '6th Stage', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status', 'Progesterone Status', 'Regional Node Examined', 'Reginol Node Positive', 'Survival Months', 'Status']


,Age,Race,Marital Status,T Stage,N Stage,6th Stage,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Survival Months,Status
0,43,"Other (American Indian/AK Native, Asian/Pacifi...",Married (including common law),T2,N3,"IIIC, Moderately differentiated",Grade II,Regional,40.0,Positive,Positive,19.0,11.0,1.0,Alive
1,47,"Other (American Indian/AK Native, Asian/Pacifi...",Married (including common law),T2,N2,"IIIA, Moderately differentiated",Grade II,Regional,45.0,Positive,Positive,25.0,9.0,2.0,Alive
2,67,White,Married (including common law),T2,N1,"IIB, Poorly differentiated",Grade III,Regional,25.0,Positive,Positive,4.0,1.0,2.0,Dead


### 2.2 Select relevant columns

SEER has 15 columns. We keep the 9 that map to our target schema and drop the rest:
- **Dropped:** Race, Marital Status, N Stage, 6th Stage, A Stage, Regional Node Examined
- These variables are either redundant (N Stage is captured by T Stage + Regional Node), not shared with METABRIC, or not clinically relevant to the survival prediction task.

In [16]:
SEER_KEEP = {
    'Age':                   'Age',
    'T Stage':               'T Stage',
    'Grade':                 'Grade',
    'Tumor Size':            'Tumor Size',
    'Estrogen Status':       'Estrogen Status',
    'Progesterone Status':   'Progesterone Status',
    'Reginol Node Positive': 'Regional Node 1',   # note: typo in source file
    'Survival Months':       'Survival Months',
    'Status':                'Status',
}

seer = seer_raw[list(SEER_KEEP.keys())].rename(columns=SEER_KEEP)
print(f'After column selection: {seer.shape}')
seer.head(3)

After column selection: (4024, 9)


,Age,T Stage,Grade,Tumor Size,Estrogen Status,Progesterone Status,Regional Node 1,Survival Months,Status
0,43,T2,Grade II,40.0,Positive,Positive,11.0,1.0,Alive
1,47,T2,Grade II,45.0,Positive,Positive,9.0,2.0,Alive
2,67,T2,Grade III,25.0,Positive,Positive,1.0,2.0,Dead


### 2.3 Drop null rows

19 rows have nulls across multiple columns (all caused by the same 19 records). We drop them entirely rather than imputing, since 19 out of 4024 is negligible (<0.5%) and the pattern suggests a data entry issue rather than missing at random.

In [17]:
print(f'Null counts before drop:\n{seer.isnull().sum()}')
seer = seer.dropna()
print(f'\nRows after dropping nulls: {len(seer):,}')

Null counts before drop:
Age                     0
T Stage                 0
Grade                   0
Tumor Size             19
Estrogen Status        19
Progesterone Status    19
Regional Node 1        19
Survival Months        19
Status                 19
dtype: int64

Rows after dropping nulls: 4,005


### 2.4 Encode categorical variables

We encode all variables as numeric for consistency with METABRIC and for use in regression and ML models.

In [18]:
# T Stage: 'T1' -> 1, 'T2' -> 2, etc.
seer['T Stage'] = seer['T Stage'].str.strip().str.extract(r'(\d)').astype(int)
print('T Stage unique values:', sorted(seer['T Stage'].unique()))

# Grade: strip leading/trailing spaces, then map
# actual values: 'Grade I', 'Grade II', 'Grade III', 'anaplastic'
grade_map = {'Grade I': 1, 'Grade II': 2, 'Grade III': 3, 'anaplastic': 3}
seer['Grade'] = seer['Grade'].str.strip().map(grade_map)
print('Grade unique values:', sorted(seer['Grade'].unique()))

# Estrogen / Progesterone Status: 'Positive' -> 1, 'Negative' -> 0
for col in ['Estrogen Status', 'Progesterone Status']:
    seer[col] = (seer[col].str.strip() == 'Positive').astype(int)
print('Estrogen unique values:', seer['Estrogen Status'].unique())

# Status: 'Alive' -> 1, 'Dead' -> 0
seer['Status'] = (seer['Status'].str.strip() == 'Alive').astype(int)
print('Status unique values:', seer['Status'].unique())

T Stage unique values: [1, 2, 3, 4]
Grade unique values: [1, 2, 3]
Estrogen unique values: [1 0]
Status unique values: [1 0]


### 2.5 Verify final SEER schema

In [19]:
seer = seer[FINAL_COLUMNS].reset_index(drop=True)
print(f'Final SEER shape: {seer.shape}')
print(f'Null values: {seer.isnull().sum().sum()}')
seer.describe().round(2)

Final SEER shape: (4005, 9)
Null values: 0


,Age,T Stage,Grade,Tumor Size,Estrogen Status,Progesterone Status,Regional Node 1,Survival Months,Status
count,4005.00,4005.00,4005.00,4005.00,4005.00,4005.00,4005.00,4005.00,4005.00
mean,53.98,1.78,2.14,30.41,0.93,0.83,4.15,71.33,0.85
std,8.95,0.76,0.63,21.08,0.25,0.38,5.11,22.87,0.36
min,30.00,1.00,1.00,1.00,0.00,0.00,1.00,1.00,0.00
25%,47.00,1.00,2.00,16.00,1.00,1.00,1.00,56.00,1.00
50%,54.00,2.00,2.00,25.00,1.00,1.00,2.00,73.00,1.00
75%,61.00,2.00,3.00,38.00,1.00,1.00,5.00,90.00,1.00
max,69.00,4.00,3.00,140.00,1.00,1.00,46.00,107.00,1.00


---
## 3. METABRIC Cleaning

METABRIC has significantly more columns (39) and requires more careful selection. The raw file comes from cBioPortal and includes genomic, histological, and clinical variables.

### 3.1 Load raw data

In [24]:
# cBioPortal files have 4 metadata rows after the header row; skip all of them
metabric_patient = pd.read_excel(METABRIC_PATIENT_PATH)
metabric_patient = metabric_patient.iloc[4:].reset_index(drop=True)

metabric_sample = pd.read_csv(METABRIC_SAMPLE_PATH, sep='\t', skiprows=[1, 2, 3, 4])

metabric_raw = metabric_patient.merge(
    metabric_sample,
    on='#Patient Identifier',
    how='inner'
)

print(f'Patient file:  {metabric_patient.shape}')
print(f'Sample file:   {metabric_sample.shape}')
print(f'Merged:        {metabric_raw.shape}')
print(f'\nAll columns:\n{list(metabric_raw.columns)}')
metabric_raw.head(3)

Patient file:  (2509, 24)
Sample file:   (2509, 13)
Merged:        (2509, 36)

All columns:
['#Patient Identifier', 'Lymph nodes examined positive', 'Nottingham prognostic index', 'Cellularity', 'Chemotherapy', 'Cohort', 'ER status measured by IHC', 'HER2 status measured by SNP6', 'Hormone Therapy', 'Inferred Menopausal State', 'Sex', 'Integrative Cluster', 'Age at Diagnosis', 'Overall Survival (Months)', 'Overall Survival Status', 'Pam50 + Claudin-low subtype', '3-Gene classifier subtype', "Patient's Vital Status", 'Primary Tumor Laterality', 'Radio Therapy', 'Tumor Other Histologic Subtype', 'Type of Breast Surgery', 'Relapse Free Status (Months)', 'Relapse Free Status', 'Sample Identifier', 'Cancer Type', 'Cancer Type Detailed', 'ER Status', 'HER2 Status', 'Neoplasm Histologic Grade', 'Oncotree Code', 'PR Status', 'Sample Type', 'Tumor Size', 'Tumor Stage', 'TMB (nonsynonymous)']


,#Patient Identifier,Lymph nodes examined positive,Nottingham prognostic index,Cellularity,Chemotherapy,Cohort,ER status measured by IHC,HER2 status measured by SNP6,Hormone Therapy,Inferred Menopausal State,...,Cancer Type Detailed,ER Status,HER2 Status,Neoplasm Histologic Grade,Oncotree Code,PR Status,Sample Type,Tumor Size,Tumor Stage,TMB (nonsynonymous)
0,MB-0000,10,6044,NaN,NO,1,Positve,NEUTRAL,YES,Post,...,Breast Invasive Ductal Carcinoma,Positive,Negative,3.0,IDC,Negative,Primary,22.0,2.0,0.000000
1,MB-0002,0,4.02,High,NO,1,Positve,NEUTRAL,YES,Pre,...,Breast Invasive Ductal Carcinoma,Positive,Negative,3.0,IDC,Positive,Primary,10.0,1.0,2.615035
2,MB-0005,1,4.03,High,YES,1,Positve,NEUTRAL,YES,Pre,...,Breast Invasive Ductal Carcinoma,Positive,Negative,2.0,IDC,Positive,Primary,15.0,2.0,2.615035


### 3.2 Select relevant columns

We keep only the columns that map to the 9-variable target schema. All genomic variables (HER2 SNP6, Pam50 subtype, TMB, Mutation Count, Integrative Cluster, etc.) are dropped — they are outside the scope of this clinical variable analysis and are not available in SEER.

In [25]:
METABRIC_KEEP = {
    'Age at Diagnosis':          'Age',
    'Tumor Stage':               'T Stage',
    'Neoplasm Histologic Grade': 'Grade',
    'Tumor Size':                'Tumor Size',
    'ER Status':                 'Estrogen Status',
    'PR Status':                 'Progesterone Status',
    'Lymph nodes examined positive': 'Regional Node 1',
    'Overall Survival (Months)': 'Survival Months',
    'Overall Survival Status':   'Status',
}

metabric = metabric_raw[list(METABRIC_KEEP.keys())].rename(columns=METABRIC_KEEP)
print(f'After column selection: {metabric.shape}')
metabric.head(3)

After column selection: (2509, 9)


,Age,T Stage,Grade,Tumor Size,Estrogen Status,Progesterone Status,Regional Node 1,Survival Months,Status
0,75.65,2.0,3.0,22.0,Positive,Negative,10,140.5,0:LIVING
1,43.19,1.0,3.0,10.0,Positive,Positive,0,84.6333333333333,0:LIVING
2,48.87,2.0,2.0,15.0,Positive,Positive,1,163.7,1:DECEASED


### 3.3 Inspect and handle nulls

In [26]:
print(f'Null counts:\n{metabric.isnull().sum()}')
print(f'\nTotal rows before drop: {len(metabric):,}')
metabric = metabric.dropna()
print(f'Total rows after drop:  {len(metabric):,}')

Null counts:
Age                     11
T Stage                721
Grade                  121
Tumor Size             149
Estrogen Status         40
Progesterone Status    529
Regional Node 1        266
Survival Months        528
Status                 528
dtype: int64

Total rows before drop: 2,509
Total rows after drop:  1,354


### 3.4 Encode categorical variables

In [27]:
# T Stage: already numeric in METABRIC — just ensure integer type
metabric['T Stage'] = pd.to_numeric(metabric['T Stage'], errors='coerce').astype('Int64')
print('T Stage unique values:', sorted(metabric['T Stage'].dropna().unique()))

# Grade: already numeric — ensure integer type
metabric['Grade'] = pd.to_numeric(metabric['Grade'], errors='coerce').astype('Int64')
print('Grade unique values:', sorted(metabric['Grade'].dropna().unique()))

# Estrogen / Progesterone Status: 'Positive' -> 1, 'Negative' -> 0
for col in ['Estrogen Status', 'Progesterone Status']:
    metabric[col] = (metabric[col].str.strip() == 'Positive').astype(int)
print('Estrogen unique values:', metabric['Estrogen Status'].unique())

# Status: '0:LIVING' -> 1 (Alive), '1:DECEASED' -> 0 (Dead)
# Note: METABRIC uses the opposite numeric convention to what we want
metabric['Status'] = (metabric['Status'].str.startswith('0')).astype(int)
print('Status unique values:', metabric['Status'].unique())

T Stage unique values: [0, 1, 2, 3, 4]
Grade unique values: [1, 2, 3]
Estrogen unique values: [1 0]
Status unique values: [1 0]


### 3.5 Verify final METABRIC schema

In [28]:
metabric = metabric[FINAL_COLUMNS].reset_index(drop=True)
print(f'Final METABRIC shape: {metabric.shape}')
print(f'Null values: {metabric.isnull().sum().sum()}')
metabric.describe().round(2)

Final METABRIC shape: (1354, 9)
Null values: 0


,T Stage,Grade,Tumor Size,Estrogen Status,Progesterone Status,Status
count,1354.0,1354.0,1354.00,1354.00,1354.00,1354.00
mean,1.76,2.44,25.86,0.77,0.52,0.44
std,0.62,0.64,14.97,0.42,0.50,0.50
min,0.0,1.0,1.00,0.00,0.00,0.00
25%,1.0,2.0,17.00,1.00,0.00,0.00
50%,2.0,3.0,22.00,1.00,1.00,0.00
75%,2.0,3.0,30.00,1.00,1.00,1.00
max,4.0,3.0,180.00,1.00,1.00,1.00


---
## 4. Validation

Before exporting, we verify that both datasets share the same schema, have no nulls, and that encoded values fall within expected ranges.

In [29]:
def validate(df, label):
    errors = []

    # Schema check
    if list(df.columns) != FINAL_COLUMNS:
        errors.append(f'Column mismatch: {list(df.columns)}')

    # Null check
    nulls = df.isnull().sum().sum()
    if nulls > 0:
        errors.append(f'{nulls} null values remaining')

    # Range checks
    checks = {
        'T Stage':             (df['T Stage'].between(0, 4).all(),     'T Stage out of range [0-4]'),
        'Grade':               (df['Grade'].between(1, 3).all(),       'Grade out of range [1-3]'),
        'Estrogen Status':     (df['Estrogen Status'].isin([0,1]).all(),'Estrogen Status not binary'),
        'Progesterone Status': (df['Progesterone Status'].isin([0,1]).all(), 'Progesterone Status not binary'),
        'Status':              (df['Status'].isin([0,1]).all(),         'Status not binary'),
    }
    for _, (passed, msg) in checks.items():
        if not passed:
            errors.append(msg)

    if errors:
        print(f'FAIL [{label}]:')
        for e in errors: print(f'  - {e}')
    else:
        print(f'PASS [{label}]: {len(df):,} rows, {df.shape[1]} columns, 0 nulls, all ranges valid')

validate(seer,     'SEER')
validate(metabric, 'METABRIC')

PASS [SEER]: 4,005 rows, 9 columns, 0 nulls, all ranges valid
PASS [METABRIC]: 1,354 rows, 9 columns, 0 nulls, all ranges valid


In [30]:
# Side-by-side descriptive stats for a final sanity check
print('=== SEER ===')
print(seer.describe().round(2).to_string())
print('\n=== METABRIC ===')
print(metabric.describe().round(2).to_string())

=== SEER ===
           Age  T Stage    Grade  Tumor Size  Estrogen Status  Progesterone Status  Regional Node 1  Survival Months   Status
count  4005.00  4005.00  4005.00     4005.00          4005.00              4005.00          4005.00          4005.00  4005.00
mean     53.98     1.78     2.14       30.41             0.93                 0.83             4.15            71.33     0.85
std       8.95     0.76     0.63       21.08             0.25                 0.38             5.11            22.87     0.36
min      30.00     1.00     1.00        1.00             0.00                 0.00             1.00             1.00     0.00
25%      47.00     1.00     2.00       16.00             1.00                 1.00             1.00            56.00     1.00
50%      54.00     2.00     2.00       25.00             1.00                 1.00             2.00            73.00     1.00
75%      61.00     2.00     3.00       38.00             1.00                 1.00             5.00      

**Sanity check observations:**
- SEER: 4005 rows, METABRIC: 1354 rows
- SEER Status mean ~0.85 (85% alive), METABRIC ~0.44 (44% alive) — matches paper
- Survival Months max: SEER ~107, METABRIC ~351 — longer follow-up window in METABRIC confirmed
- All binary variables (Estrogen, Progesterone, Status) contain only 0 and 1

---
## 5. Export

Write both cleaned datasets to a single Excel workbook with named sheets. This is the file consumed by all downstream notebooks.

In [31]:
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    seer.to_excel(writer,     sheet_name='SEER',     index=False)
    metabric.to_excel(writer, sheet_name='METABRIC', index=False)

print(f'Exported to {OUTPUT_PATH}')
print(f'  Sheet SEER:     {len(seer):,} rows')
print(f'  Sheet METABRIC: {len(metabric):,} rows')

Exported to data/clean_data_breast_cancer.xlsx
  Sheet SEER:     4,005 rows
  Sheet METABRIC: 1,354 rows


---
## Cleaning decisions log

| Decision | Rationale |
|---|---|
| Dropped Race, Marital Status (SEER) | Not shared with METABRIC; outside scope of clinical variable analysis |
| Dropped N Stage, 6th Stage, A Stage (SEER) | Redundant with T Stage + Regional Node |
| Dropped genomic variables (METABRIC) | Not available in SEER; outside scope |
| Dropped 19 null rows in SEER | <0.5% of data; all nulls in same records suggesting entry issue |
| Mapped 'anaplastic' Grade to 3 | Anaplastic is a high-grade variant — most aggressive category |
| Status encoded as 1=Alive, 0=Dead | Positive class = survival, which is the outcome of interest |
| METABRIC Status: '0:LIVING' -> 1 | cBioPortal uses opposite convention — corrected for consistency |
| Age rounded to integer (METABRIC) | Source has decimal ages; integer is consistent with SEER |